# Notebook 5 — MongoDB schema, loading, indexed queries, and comparison with Spark

This notebook covers the **MongoDB and query comparison** part of the project.

It uses the processed analytical base dataset from **Notebook 3** to:

- define a simple and clear **MongoDB schema**
- load processed data into **MongoDB**
- create **indexes** for relevant query fields
- run representative **MongoDB queries**
- compare MongoDB results and timing with **Spark SQL / PySpark**

## Assumptions

- **Notebook 3** has already been run successfully.
- The processed dataset exists at `data/processed/video_games_analytical_base/`.
- MongoDB is available either locally or through **MongoDB Atlas**.
- This notebook uses a **referenced model**:
  - one collection for **products**
  - one collection for **reviews**

## Expected columns from Notebook 3

- `parent_asin`
- `user_id`
- `product_title`
- `main_category`
- `categories_clean`
- `category_count`
- `store_clean`
- `price_value`
- `rating`
- `verified_purchase`
- `helpful_vote`
- `review_text_length`
- `review_timestamp_ms`
- `review_ts`
- `review_date`
- `review_year`
- `review_month`
- `review_year_month`

## Section 1: Imports

In [ ]:
# Uncomment and run this cell only if pymongo is not installed
# %pip install pymongo

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from time import perf_counter

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

from pymongo import MongoClient, InsertOne, ASCENDING, DESCENDING

## Section 2: Environment Setup

In [3]:
import os

if os.name == "nt":
    hadoop_home = Path(r"C:\hadoop")
    if hadoop_home.exists():
        os.environ["HADOOP_HOME"] = str(hadoop_home)
        hadoop_bin = str(hadoop_home / "bin")
        if hadoop_bin not in os.environ["PATH"]:
            os.environ["PATH"] = os.environ["PATH"] + ";" + hadoop_bin
        print(f"Windows detected. Using HADOOP_HOME={hadoop_home}")
    else:
        print(
            "Windows detected, but C:\\hadoop was not found. "
            "If Parquet read/write fails, configure Hadoop / winutils as in the earlier notebooks."
        )
else:
    print("Non-Windows OS detected. Skipping HADOOP_HOME setup.")

Non-Windows OS detected. Skipping HADOOP_HOME setup.


## Section 3: Paths, MongoDB Settings, and Spark Session

In [4]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "README.md").exists()
            and (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Make sure the notebook is inside the project repository."
    )

PROJECT_ROOT = find_project_root()
BASE_PATH = PROJECT_ROOT / "data" / "processed" / "video_games_analytical_base"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert BASE_PATH.exists(), f"Missing processed dataset directory: {BASE_PATH}"

# MongoDB settings
MONGO_URI = "mongodb://localhost:27017"
MONGO_DB_NAME = "amazon_video_games_group14"
PRODUCTS_COLLECTION = "products"
REVIEWS_COLLECTION = "reviews"

# Development toggles
DROP_EXISTING_COLLECTIONS = True
MAX_PRODUCT_DOCS = None   # Example: 5000 for a quick test
MAX_REVIEW_DOCS = None    # Example: 100000 for a faster development run
BATCH_SIZE = 5000

spark = (
    SparkSession.builder
    .appName("Notebook_5_MongoDB_Amazon_Video_Games")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.driver.memory", "3g")
    .config("spark.memory.fraction", "0.8")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Working directory: {Path.cwd().resolve()}")
print(f"Project root:      {PROJECT_ROOT}")
print(f"Processed path:    {BASE_PATH}")
print(f"Results path:      {RESULTS_DIR}")
print(f"Mongo URI:         {MONGO_URI}")
print(f"Mongo database:    {MONGO_DB_NAME}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 13:09:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Working directory: /Users/lenobach/git/HAMK/BDA/big-data-video-games-reviews/notebooks
Project root:      /Users/lenobach/git/HAMK/BDA/big-data-video-games-reviews
Processed path:    /Users/lenobach/git/HAMK/BDA/big-data-video-games-reviews/data/processed/video_games_analytical_base
Results path:      /Users/lenobach/git/HAMK/BDA/big-data-video-games-reviews/results
Mongo URI:         mongodb://localhost:27017
Mongo database:    amazon_video_games_group14


## Section 4: Load the Processed Analytical Base Dataset

In [5]:
base_df = spark.read.parquet(str(BASE_PATH))

print(f"Rows: {base_df.count():,}")
print(f"Distinct products: {base_df.select('parent_asin').distinct().count():,}")
print(f"Distinct users: {base_df.select('user_id').distinct().count():,}")

base_df.printSchema()
base_df.show(5, truncate=False)

Rows: 4,624,615


Distinct products: 137,249


Distinct users: 2,766,656
root
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- categories_clean: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- category_count: integer (nullable = true)
 |-- store_clean: string (nullable = true)
 |-- price_value: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- review_text_length: integer (nullable = true)
 |-- review_timestamp_ms: long (nullable = true)
 |-- review_ts: timestamp (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)
 |-- review_year_month: timestamp (nullable = true)
 |-- has_price: boolean (nullable = true)
 |-- has_categories: boolean (nullable = true)

+-----------+----------

## Section 5: Cache the Base DataFrame

In [6]:
base_df = base_df.persist(StorageLevel.MEMORY_AND_DISK)
base_df.count()  # materialize cache

print("Base DataFrame cached.")

Base DataFrame cached.


## Section 6: MongoDB Schema Choice

We use a **referenced MongoDB schema** with two collections:

### `products`
One document per product.

Main fields:
- `_id` = `parent_asin`
- `product_title`
- `main_category`
- `categories_clean`
- `category_count`
- `store_clean`
- `price_value`

### `reviews`
One document per review.

Main fields:
- `_id` = generated stable hash
- `parent_asin` (reference to product `_id`)
- `user_id`
- `rating`
- `verified_purchase`
- `helpful_vote`
- `review_text_length`
- `review_timestamp_ms`
- `review_ts`
- `review_date`
- `review_year`
- `review_month`
- `review_year_month`

This design is simple and practical for this project.  
It avoids putting all reviews inside one product document, which would not scale well for products with many reviews.

In [7]:
products_df = (
    base_df
    .select(
        F.col("parent_asin").alias("_id"),
        "product_title",
        "main_category",
        "categories_clean",
        "category_count",
        "store_clean",
        "price_value"
    )
    .dropDuplicates(["_id"])
)

reviews_df_raw = (
    base_df
    .withColumn("review_date", F.col("review_date").cast("string"))
    .withColumn("review_year_month", F.date_format(F.col("review_year_month"), "yyyy-MM"))
    .withColumn(
        "_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("parent_asin"), F.lit("")),
                F.coalesce(F.col("user_id"), F.lit("")),
                F.coalesce(F.col("rating").cast("string"), F.lit("")),
                F.coalesce(F.col("verified_purchase").cast("string"), F.lit("")),
                F.coalesce(F.col("helpful_vote").cast("string"), F.lit("")),
                F.coalesce(F.col("review_text_length").cast("string"), F.lit("")),
                F.coalesce(F.col("review_timestamp_ms").cast("string"), F.lit(""))
            ),
            256
        )
    )
    .select(
        "_id",
        "parent_asin",
        "user_id",
        "rating",
        "verified_purchase",
        "helpful_vote",
        "review_text_length",
        "review_timestamp_ms",
        "review_ts",
        "review_date",
        "review_year",
        "review_month",
        "review_year_month"
    )
)

reviews_df = reviews_df_raw.dropDuplicates(["_id"])

products_count = products_df.count()
reviews_raw_count = reviews_df_raw.count()
reviews_final_count = reviews_df.count()

print(f"Products rows:            {products_count:,}")
print(f"Review rows before dedup: {reviews_raw_count:,}")
print(f"Review rows after dedup:  {reviews_final_count:,}")
print(f"Duplicate reviews removed: {reviews_raw_count - reviews_final_count:,}")

reviews_df.createOrReplaceTempView("video_game_reviews_dedup")
print("Created temp view: video_game_reviews_dedup")

Products rows:            137,249
Review rows before dedup: 4,624,615
Review rows after dedup:  4,570,965
Duplicate reviews removed: 53,650
Created temp view: video_game_reviews_dedup


## Section 7: Validate the MongoDB-Ready DataFrames

In [8]:
products_df.printSchema()
reviews_df.printSchema()

products_df.show(5, truncate=False)
reviews_df.show(5, truncate=False)

root
 |-- _id: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- categories_clean: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- category_count: integer (nullable = true)
 |-- store_clean: string (nullable = true)
 |-- price_value: double (nullable = true)

root
 |-- _id: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- review_text_length: integer (nullable = true)
 |-- review_timestamp_ms: long (nullable = true)
 |-- review_ts: timestamp (nullable = true)
 |-- review_date: string (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)
 |-- review_year_month: string (nullable = true)



+----------+---------------------------------------------------------------------------------------------+-------------+-----------------------------------------------+--------------+--------------------------------+-----------+
|_id       |product_title                                                                                |main_category|categories_clean                               |category_count|store_clean                     |price_value|
+----------+---------------------------------------------------------------------------------------------+-------------+-----------------------------------------------+--------------+--------------------------------+-----------+
|0063052164|Stranger Planet AUTOGRAPHED / SIGNED BOOK                                                    |Video Games  |[Video Games, PlayStation 4]                   |2             |splins                          |19.99      |
|0152049495|Dogzilla                                                                

+----------------------------------------------------------------+-----------+----------------------------+------+-----------------+------------+------------------+-------------------+-----------------------+-----------+-----------+------------+-----------------+
|_id                                                             |parent_asin|user_id                     |rating|verified_purchase|helpful_vote|review_text_length|review_timestamp_ms|review_ts              |review_date|review_year|review_month|review_year_month|
+----------------------------------------------------------------+-----------+----------------------------+------+-----------------+------------+------------------+-------------------+-----------------------+-----------+-----------+------------+-----------------+
|00001642f4cb4f141212dd981f2b6ed2122705c4a3fada83aa0b10bf50a563e4|B08MBPWRSF |AHHCZUQUHCFCP7MDCYIY2DD3G6LA|3.0   |true             |0           |254               |1619675856044      |2021-04-29 05:57:36.044|

## Section 8: Connect to MongoDB

In [9]:
from pymongo.errors import ServerSelectionTimeoutError

try:
    mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    mongo_client.admin.command("ping")
except ServerSelectionTimeoutError as e:
    raise ConnectionError(
        "Could not connect to MongoDB. Make sure MongoDB is running, or update MONGO_URI."
    ) from e

db = mongo_client[MONGO_DB_NAME]

if DROP_EXISTING_COLLECTIONS:
    db.drop_collection(PRODUCTS_COLLECTION)
    db.drop_collection(REVIEWS_COLLECTION)
    print("Existing collections dropped.")

products_col = db[PRODUCTS_COLLECTION]
reviews_col = db[REVIEWS_COLLECTION]

print("MongoDB connection successful.")
print("Collections ready:", db.list_collection_names())

Existing collections dropped.
MongoDB connection successful.
Collections ready: []


## Section 9: Helper Function for Batch Insertion

In [10]:
def batched_insert_from_spark(df, collection, batch_size=5000):
    total_inserted = 0
    batch = []

    for row in df.toLocalIterator():
        batch.append(InsertOne(row.asDict(recursive=True)))

        if len(batch) >= batch_size:
            result = collection.bulk_write(batch, ordered=False)
            total_inserted += result.inserted_count
            batch = []

            if total_inserted % 250000 == 0:
                print(f"Inserted so far into {collection.name}: {total_inserted:,}")

    if batch:
        result = collection.bulk_write(batch, ordered=False)
        total_inserted += result.inserted_count

    return total_inserted

## Section 10: Load the Products Collection

In [11]:
products_to_write = products_df
if MAX_PRODUCT_DOCS is not None:
    products_to_write = products_to_write.limit(MAX_PRODUCT_DOCS)

start = perf_counter()
products_inserted = batched_insert_from_spark(
    products_to_write,
    products_col,
    batch_size=BATCH_SIZE
)
products_load_time = perf_counter() - start

print(f"Inserted products: {products_inserted:,}")
print(f"Products load time: {products_load_time:.2f} seconds")
print(f"MongoDB products count: {products_col.count_documents({}):,}")

Inserted products: 137,249
Products load time: 4.76 seconds
MongoDB products count: 137,249


## Section 11: Load the Reviews Collection

In [12]:
reviews_to_write = reviews_df
if MAX_REVIEW_DOCS is not None:
    reviews_to_write = reviews_to_write.limit(MAX_REVIEW_DOCS)

start = perf_counter()
reviews_inserted = batched_insert_from_spark(
    reviews_to_write,
    reviews_col,
    batch_size=BATCH_SIZE
)
reviews_load_time = perf_counter() - start

print(f"Inserted reviews: {reviews_inserted:,}")
print(f"Reviews load time: {reviews_load_time:.2f} seconds")
print(f"MongoDB reviews count: {reviews_col.count_documents({}):,}")

Inserted so far into reviews: 250,000


Inserted so far into reviews: 500,000
Inserted so far into reviews: 750,000


Inserted so far into reviews: 1,000,000


Inserted so far into reviews: 1,250,000
Inserted so far into reviews: 1,500,000


Inserted so far into reviews: 1,750,000


...

Inserted so far into reviews: 2,750,000
Inserted so far into reviews: 3,000,000


Inserted so far into reviews: 3,250,000


Inserted so far into reviews: 3,500,000
Inserted so far into reviews: 3,750,000


Inserted so far into reviews: 4,000,000
Inserted so far into reviews: 4,250,000


Inserted so far into reviews: 4,500,000
Inserted reviews: 4,570,965
Reviews load time: 88.18 seconds
MongoDB reviews count: 4,570,965


## Section 12: Create Indexes

These indexes are created on fields used in the main MongoDB queries so that lookups and filtering are faster.

In [13]:
products_col.create_index([("product_title", ASCENDING)], name="idx_product_title")

reviews_col.create_index([("parent_asin", ASCENDING)], name="idx_parent_asin")
reviews_col.create_index([("rating", ASCENDING)], name="idx_rating")
reviews_col.create_index([("verified_purchase", ASCENDING)], name="idx_verified_purchase")
reviews_col.create_index([("review_year_month", ASCENDING)], name="idx_review_year_month")
reviews_col.create_index(
    [("parent_asin", ASCENDING), ("helpful_vote", DESCENDING)],
    name="idx_parent_helpful_vote"
)

print("Products indexes:")
for name, meta in products_col.index_information().items():
    print(name, "->", meta["key"])

print("\nReviews indexes:")
for name, meta in reviews_col.index_information().items():
    print(name, "->", meta["key"])

Products indexes:
_id_ -> [('_id', 1)]
idx_product_title -> [('product_title', 1)]

Reviews indexes:
_id_ -> [('_id', 1)]
idx_parent_asin -> [('parent_asin', 1)]
idx_rating -> [('rating', 1)]
idx_verified_purchase -> [('verified_purchase', 1)]
idx_review_year_month -> [('review_year_month', 1)]
idx_parent_helpful_vote -> [('parent_asin', 1), ('helpful_vote', -1)]


## Section 13: Define Query Timing Helpers and Comparison Queries

The following helper functions are used to run the same analytical questions in MongoDB and Spark and compare their execution times.

In [14]:
def time_mongo_aggregate(collection, pipeline, runs=3):
    times = []
    result_docs = None

    for _ in range(runs):
        start = perf_counter()
        result_docs = list(collection.aggregate(pipeline, allowDiskUse=True))
        times.append(perf_counter() - start)

    return pd.DataFrame(result_docs), times


def time_mongo_find(collection, query, sort=None, limit=None, projection=None, runs=3):
    times = []
    result_docs = None

    for _ in range(runs):
        start = perf_counter()

        cursor = collection.find(query, projection)
        if sort is not None:
            cursor = cursor.sort(sort)
        if limit is not None:
            cursor = cursor.limit(limit)

        result_docs = list(cursor)
        times.append(perf_counter() - start)

    return pd.DataFrame(result_docs), times


def time_spark_builder(builder, runs=3):
    times = []
    result_pdf = None

    for _ in range(runs):
        start = perf_counter()
        result_pdf = builder().toPandas()
        times.append(perf_counter() - start)

    return result_pdf, times


def summarize_timing(query_name, mongo_times, spark_times):
    return {
        "query_name": query_name,
        "mongo_avg_seconds": sum(mongo_times) / len(mongo_times),
        "mongo_min_seconds": min(mongo_times),
        "spark_avg_seconds": sum(spark_times) / len(spark_times),
        "spark_min_seconds": min(spark_times),
    }


comparison_rows = []

rating_pipeline = [
    {"$group": {"_id": "$rating", "review_count": {"$sum": 1}}},
    {"$project": {"_id": 0, "rating": "$_id", "review_count": 1}},
    {"$sort": {"rating": 1}}
]

verified_pipeline = [
    {
        "$group": {
            "_id": "$verified_purchase",
            "review_count": {"$sum": 1},
            "avg_rating": {"$avg": "$rating"},
            "avg_helpful_vote": {"$avg": "$helpful_vote"},
            "avg_review_text_length": {"$avg": "$review_text_length"}
        }
    },
    {
        "$project": {
            "_id": 0,
            "verified_purchase": "$_id",
            "review_count": 1,
            "avg_rating": 1,
            "avg_helpful_vote": 1,
            "avg_review_text_length": 1
        }
    },
    {"$sort": {"verified_purchase": 1}}
]

top_products_pipeline = [
    {
        "$group": {
            "_id": "$parent_asin",
            "review_count": {"$sum": 1},
            "avg_rating": {"$avg": "$rating"}
        }
    },
    {"$match": {"review_count": {"$gte": 100}}},
    {"$sort": {"review_count": -1, "avg_rating": -1}},
    {"$limit": 10},
    {
        "$lookup": {
            "from": PRODUCTS_COLLECTION,
            "localField": "_id",
            "foreignField": "_id",
            "as": "product"
        }
    },
    {"$unwind": {"path": "$product", "preserveNullAndEmptyArrays": True}},
    {
        "$project": {
            "_id": 0,
            "parent_asin": "$_id",
            "product_title": "$product.product_title",
            "review_count": 1,
            "avg_rating": 1
        }
    }
]


def build_spark_rating_df():
    return spark.sql(
        """
        SELECT
            rating,
            COUNT(*) AS review_count
        FROM video_game_reviews_dedup
        GROUP BY rating
        ORDER BY rating
        """
    )


def build_spark_verified_df():
    return spark.sql(
        """
        SELECT
            verified_purchase,
            COUNT(*) AS review_count,
            AVG(rating) AS avg_rating,
            AVG(helpful_vote) AS avg_helpful_vote,
            AVG(review_text_length) AS avg_review_text_length
        FROM video_game_reviews_dedup
        GROUP BY verified_purchase
        ORDER BY verified_purchase
        """
    )


def build_spark_top_products_df():
    return (
        reviews_df
        .join(
            products_df.select(
                F.col("_id").alias("parent_asin"),
                "product_title"
            ),
            on="parent_asin",
            how="left"
        )
        .groupBy("parent_asin", "product_title")
        .agg(
            F.count("*").alias("review_count"),
            F.avg("rating").alias("avg_rating")
        )
        .filter(F.col("review_count") >= 100)
        .orderBy(F.desc("review_count"), F.desc("avg_rating"))
        .limit(10)
    )

## Section 14: Query 1 — Rating Distribution

In [15]:
mongo_rating_pdf, mongo_rating_times = time_mongo_aggregate(
    reviews_col,
    rating_pipeline,
    runs=3
)

spark_rating_pdf, spark_rating_times = time_spark_builder(
    build_spark_rating_df,
    runs=3
)

mongo_rating_pdf = mongo_rating_pdf.sort_values("rating").reset_index(drop=True)
spark_rating_pdf = spark_rating_pdf.sort_values("rating").reset_index(drop=True)

comparison_rows.append(
    summarize_timing("rating_distribution", mongo_rating_times, spark_rating_times)
)

print("MongoDB result:")
display(mongo_rating_pdf)

print("Spark result:")
display(spark_rating_pdf)

MongoDB result:


,review_count,rating
0,581765,1.0
1,246928,2.0
2,336178,3.0
3,610083,4.0
4,2796011,5.0


Spark result:


,rating,review_count
0,1.0,581765
1,2.0,246928
2,3.0,336178
3,4.0,610083
4,5.0,2796011


## Section 15: Query 2 — Verified vs Non-Verified Purchase Behavior

In [16]:
mongo_verified_pdf, mongo_verified_times = time_mongo_aggregate(
    reviews_col,
    verified_pipeline,
    runs=3
)

spark_verified_pdf, spark_verified_times = time_spark_builder(
    build_spark_verified_df,
    runs=3
)

mongo_verified_pdf = mongo_verified_pdf.sort_values("verified_purchase").reset_index(drop=True)
spark_verified_pdf = spark_verified_pdf.sort_values("verified_purchase").reset_index(drop=True)

comparison_rows.append(
    summarize_timing("verified_summary", mongo_verified_times, spark_verified_times)
)

print("MongoDB result:")
display(mongo_verified_pdf)

print("Spark result:")
display(spark_verified_pdf)

MongoDB result:


,review_count,avg_rating,avg_helpful_vote,avg_review_text_length,verified_purchase
0,634347,3.731644,3.326772,801.372041,False
1,3936618,4.099302,0.891204,228.250837,True


Spark result:


,verified_purchase,review_count,avg_rating,avg_helpful_vote,avg_review_text_length
0,False,634347,3.731644,3.326772,801.372041
1,True,3936618,4.099302,0.891204,228.250837


## Section 16: Query 3 — Top Products by Review Count

In [17]:
mongo_top_products_pdf, mongo_top_products_times = time_mongo_aggregate(
    reviews_col,
    top_products_pipeline,
    runs=3
)

spark_top_products_pdf, spark_top_products_times = time_spark_builder(
    build_spark_top_products_df,
    runs=3
)

comparison_rows.append(
    summarize_timing("top_products_with_lookup", mongo_top_products_times, spark_top_products_times)
)

print("MongoDB result:")
display(mongo_top_products_pdf)

print("Spark result:")
display(spark_top_products_pdf)

MongoDB result:


,review_count,avg_rating,parent_asin,product_title
0,17893,4.649863,B01N3ASPNV,amFilm Tempered Glass Screen Protector for Nin...
1,17146,4.181325,B0BN942894,"BENGOO Stereo Pro Gaming Headset for PS4, PC, ..."
2,15426,3.977376,B077GG9D5D,DualShock 4 Wireless Controller for PlayStatio...
3,13143,4.288595,B000N5Z2L4,Xbox Live Gold: 1 Month Membership [Digital Code]
4,11960,4.493395,B0086VPUHI,Grand Theft Auto V: Premium Edition - Xbox One...
5,10141,4.659994,B004RMK5QG,PlayStation Plus: 12 Month Membership [Digital...
6,9323,4.482141,B07YBXFDYN,Skyrim VR - PlayStation 4
7,8014,4.220115,B00BGA9WK2,PlayStation 4 500GB Console [Old Model][Discon...
8,7782,3.363788,B07V8YSBFG,"Roblox Digital Gift Code for 1,200 Robux [Rede..."
9,7465,4.427060,B07CRX5X9T,UtechSmart Venus Pro RGB Wireless MMO Gaming M...


Spark result:


,parent_asin,product_title,review_count,avg_rating
0,B01N3ASPNV,amFilm Tempered Glass Screen Protector for Nin...,17893,4.649863
1,B0BN942894,"BENGOO Stereo Pro Gaming Headset for PS4, PC, ...",17146,4.181325
2,B077GG9D5D,DualShock 4 Wireless Controller for PlayStatio...,15426,3.977376
3,B000N5Z2L4,Xbox Live Gold: 1 Month Membership [Digital Code],13143,4.288595
4,B0086VPUHI,Grand Theft Auto V: Premium Edition - Xbox One...,11960,4.493395
5,B004RMK5QG,PlayStation Plus: 12 Month Membership [Digital...,10141,4.659994
6,B07YBXFDYN,Skyrim VR - PlayStation 4,9323,4.482141
7,B00BGA9WK2,PlayStation 4 500GB Console [Old Model][Discon...,8014,4.220115
8,B07V8YSBFG,"Roblox Digital Gift Code for 1,200 Robux [Rede...",7782,3.363788
9,B07CRX5X9T,UtechSmart Venus Pro RGB Wireless MMO Gaming M...,7465,4.427060


## Section 17: Query 4 — Top Helpful Reviews for One Example Product

In [18]:
example_parent_asin = mongo_top_products_pdf.iloc[0]["parent_asin"]
example_product_title = mongo_top_products_pdf.iloc[0]["product_title"]

print("Example product for targeted lookup:")
print("parent_asin:", example_parent_asin)
print("product_title:", example_product_title)

Example product for targeted lookup:
parent_asin: B01N3ASPNV
product_title: amFilm Tempered Glass Screen Protector for Nintendo Switch 2017 (2-Pack)


In [19]:
def build_spark_targeted_reviews_df():
    return (
        reviews_df
        .filter(F.col("parent_asin") == example_parent_asin)
        .select(
            "parent_asin",
            "user_id",
            "rating",
            "helpful_vote",
            "verified_purchase",
            "review_year_month"
        )
        .orderBy(F.desc("helpful_vote"))
        .limit(20)
    )

mongo_targeted_pdf, mongo_targeted_times = time_mongo_find(
    reviews_col,
    query={"parent_asin": example_parent_asin},
    sort=[("helpful_vote", -1)],
    limit=20,
    projection={
        "_id": 0,
        "parent_asin": 1,
        "user_id": 1,
        "rating": 1,
        "helpful_vote": 1,
        "verified_purchase": 1,
        "review_year_month": 1
    },
    runs=3
)

spark_targeted_pdf, spark_targeted_times = time_spark_builder(
    build_spark_targeted_reviews_df,
    runs=3
)

comparison_rows.append(
    summarize_timing("targeted_product_lookup", mongo_targeted_times, spark_targeted_times)
)

print("MongoDB result:")
display(mongo_targeted_pdf)

print("Spark result:")
display(spark_targeted_pdf)

MongoDB result:


,parent_asin,user_id,rating,verified_purchase,helpful_vote,review_year_month
0,B01N3ASPNV,AHHXIA3VFUZHFC6KEBBH6ADVA6VQ,1.0,True,1027,2019-01
1,B01N3ASPNV,AHZLH3PKL62PH7444T5F2NJDRF7A,5.0,True,542,2017-03
2,B01N3ASPNV,AGJTO5O3BPXSLJPXDI42BT75VHRQ,5.0,True,513,2018-03
3,B01N3ASPNV,AE7BI56AUWWRXIZT2GTZ5MHC7PKA,5.0,True,366,2018-03
4,B01N3ASPNV,AEDSNIEIVHXPPMBMXSLJDEI7EFPA,5.0,True,237,2017-07
5,B01N3ASPNV,AFUG4QIR33JYN6KBPBTG2ZL4BVVQ,1.0,True,185,2018-09
6,B01N3ASPNV,AG46CWJGT36TUWUYPM3OESZODEKA,5.0,True,113,2017-03
7,B01N3ASPNV,AGPCVLOZQWV6O3G5HDDD5WIMD6CA,5.0,True,108,2017-03
8,B01N3ASPNV,AETJ3NGNAX64MY7D2XVHVTBR2MZA,5.0,True,61,2017-03
9,B01N3ASPNV,AEMTWSK54VTP5VFYUJWAWM7TFYUA,4.0,True,58,2017-03


Spark result:


,parent_asin,user_id,rating,helpful_vote,verified_purchase,review_year_month
0,B01N3ASPNV,AHHXIA3VFUZHFC6KEBBH6ADVA6VQ,1.0,1027,True,2019-01
1,B01N3ASPNV,AHZLH3PKL62PH7444T5F2NJDRF7A,5.0,542,True,2017-03
2,B01N3ASPNV,AGJTO5O3BPXSLJPXDI42BT75VHRQ,5.0,513,True,2018-03
3,B01N3ASPNV,AE7BI56AUWWRXIZT2GTZ5MHC7PKA,5.0,366,True,2018-03
4,B01N3ASPNV,AEDSNIEIVHXPPMBMXSLJDEI7EFPA,5.0,237,True,2017-07
5,B01N3ASPNV,AFUG4QIR33JYN6KBPBTG2ZL4BVVQ,1.0,185,True,2018-09
6,B01N3ASPNV,AG46CWJGT36TUWUYPM3OESZODEKA,5.0,113,True,2017-03
7,B01N3ASPNV,AGPCVLOZQWV6O3G5HDDD5WIMD6CA,5.0,108,True,2017-03
8,B01N3ASPNV,AETJ3NGNAX64MY7D2XVHVTBR2MZA,5.0,61,True,2017-03
9,B01N3ASPNV,AEMTWSK54VTP5VFYUJWAWM7TFYUA,4.0,58,True,2017-03


## Section 18: Timing Summary

This table summarizes the average and minimum execution times for the MongoDB and Spark versions of each query.

In [20]:
timing_summary_df = (
    pd.DataFrame(comparison_rows)
    .sort_values("query_name")
    .reset_index(drop=True)
)

timing_summary_df

,query_name,mongo_avg_seconds,mongo_min_seconds,spark_avg_seconds,spark_min_seconds
0,rating_distribution,0.824777,0.767354,3.196325,2.808238
1,targeted_product_lookup,0.009515,0.000702,4.838717,4.678380
2,top_products_with_lookup,2.333122,1.963253,4.951125,4.453331
3,verified_summary,1.926726,1.764860,3.182683,2.947759


## Section 19: Save Selected Outputs

In [21]:
timing_summary_path = RESULTS_DIR / "notebook5_timing_summary.csv"
mongo_top_products_path = RESULTS_DIR / "notebook5_top_products_mongo.csv"
spark_top_products_path = RESULTS_DIR / "notebook5_top_products_spark.csv"

timing_summary_df.to_csv(timing_summary_path, index=False)
mongo_top_products_pdf.to_csv(mongo_top_products_path, index=False)
spark_top_products_pdf.to_csv(spark_top_products_path, index=False)

print("Saved:")
print(" -", timing_summary_path)
print(" -", mongo_top_products_path)
print(" -", spark_top_products_path)

Saved:
 - /Users/lenobach/git/HAMK/BDA/big-data-video-games-reviews/results/notebook5_timing_summary.csv
 - /Users/lenobach/git/HAMK/BDA/big-data-video-games-reviews/results/notebook5_top_products_mongo.csv
 - /Users/lenobach/git/HAMK/BDA/big-data-video-games-reviews/results/notebook5_top_products_spark.csv


## Section 20: Summary

In this notebook, we completed the MongoDB part of the project by taking the processed analytical base dataset from Notebook 3, transforming it into MongoDB-ready product and review documents, and loading it into MongoDB using a referenced schema with separate `products` and `reviews` collections.

A small number of duplicate review rows were identified in the processed dataset, so the review data was deduplicated before loading into MongoDB. This made the MongoDB results and the Spark comparison queries consistent.

We then created indexes for the main query fields and ran representative MongoDB queries for:

- rating distribution
- verified vs non-verified purchase behavior
- top products by review count
- top helpful reviews for one example product

The same analytical questions were also run with Spark so that query results and timing could be compared fairly on the same deduplicated review dataset.

Overall, the results show that:

- **MongoDB** is well suited for indexed lookups, document storage, and operational-style queries
- **Spark** is well suited for large-scale transformation pipelines and analytical processing
- in this project, **Spark and MongoDB complement each other** rather than replacing one another